<a href="https://colab.research.google.com/github/netsetos/agentic-ai-weekend-gcp-learners/blob/main/module-12-production-deploy/lesson-12.7-keyless-cicd/notebooks/GCP_Capstone_12.7_KeylessCICD.ipynb" target="_blank"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 12.7 Keyless CI/CD — The Release Is a Candidate
**Netsetos GenAI Engineering — GCP Capstone** · Module 12 · rebuilt on the live lane, 10 September 2026

DocuMind is built, observable, has a face, ingests real documents and is guarded. This lesson ships it - without a credential in a repository secret, and on the lane that exists. Every change on this lane since 8 September has cleared the same way: a revision that takes no traffic, the 64-row gate against it, then traffic - the tuned model, the gateway, the guard. A release is that path with a new image at the front: `make build`, `make release-candidate`, `make eval-live` on the candidate, a person, `make promote`; and `make rollback` is the same flip the other way. This notebook runs the offline gate from the clone, reads the candidate and runs the live gate on it, reads the Workload Identity provider and the deploy account's roles back, parses the workflow's lean jobs, prints the flip and the rollback with their revisions (and runs them behind a guard), and reads the full profile's Cloud Deploy pipeline from the clone. The first version's tables stay where they earn their place; the eight heredocs at the end are the kit's source.


## Setup


In [ ]:
!pip install -q google-genai==2.22.0 google-cloud-storage==3.13.1 requests==2.34.2

from google.colab import auth
auth.authenticate_user()

PROJECT_ID = "documind-ai-YOUR-ID"   # CHANGE THIS: the project the lane runs in (make up, lesson 4.8)
REGION     = "us-central1"
TENANT     = "acme"
KIT        = "/content/agentic-ai-weekend-gcp-learners"   # the kit: deploy/shared is the tool layer every lesson on the lane imports
BRANCH     = "main"        # the learner repo's branch: the notebooks and the kit (deploy/) ship there together

import os, subprocess, sys
import google.auth
from google.auth.transport.requests import AuthorizedSession
from google import genai
from google.genai import types

if not os.path.isdir(KIT):
    subprocess.run(["git", "clone", "--depth", "1", "-q", "-b", BRANCH,
                    "https://github.com/netsetos/agentic-ai-weekend-gcp-learners", KIT], check=True)
sys.path.insert(0, f"{KIT}/deploy")                   # `from shared import ...` - the same layer every service imports

# The lane's URLs are deterministic: service name + project NUMBER (eventarc.tf builds them the same way).
creds, _ = google.auth.default()
NUMBER = AuthorizedSession(creds).get(
    f"https://cloudresourcemanager.googleapis.com/v1/projects/{PROJECT_ID}").json()["projectNumber"]
API_URL       = f"https://documind-api-{NUMBER}.{REGION}.run.app"
UPLOAD_BUCKET = f"{PROJECT_ID}-uploads"     # storage.tf: the bucket eventarc.tf watches - the corpus, media included
MEDIA_BUCKET  = f"{PROJECT_ID}-media"       # storage.tf: generated assets, 30-day lifecycle (a cache, not a record)
DATASETS      = f"{PROJECT_ID}-datasets"    # storage.tf (Module 10): the frozen tuning dataset and 10.5's GGUF
GATEWAY_URL   = f"https://documind-gateway-{NUMBER}.{REGION}.run.app"   # 11.3: LiteLLM on Cloud Run, behind IAM (make deploy-gateway)
SLM_URL       = f"https://documind-slm-{NUMBER}.{REGION}.run.app"       # 11.4: Ollama on an L4, min-instances 0 (make deploy-slm)
os.environ.update({
    "GOOGLE_CLOUD_PROJECT": PROJECT_ID,
    "GOOGLE_CLOUD_LOCATION": "global",              # Gemini 3.x generation is served from the global endpoint
    "GOOGLE_GENAI_USE_VERTEXAI": "TRUE",
    "DOCUMIND_PROFILE": "gcp",
    "RAG_API_URL": API_URL,
    "RAG_TIMEOUT_S": "90",                          # 7.2's finding: a cold API takes longer than the default 20 s
    # A notebook has no metadata server to be anyone with: the kit mints its ID tokens AS this roster
    # member (7.1). On Cloud Run the service's own account is the identity and nothing is set.
    "DOCUMIND_IMPERSONATE_SA": f"documind-ui-sa@{PROJECT_ID}.iam.gserviceaccount.com",
})
MEMBER_SA   = os.environ["DOCUMIND_IMPERSONATE_SA"]
OUTSIDER_SA = f"documind-outsider-sa@{PROJECT_ID}.iam.gserviceaccount.com"   # IAM admits it, no roster does (4.8, 7.2)

from shared import documind_tools    # THE one retrieve(). Imported, never pasted - the contract gate fails a paste.
gen = genai.Client(enterprise=True, project=PROJECT_ID, location="global")   # every generate_content in this lesson
CANDIDATE_URL = f"https://candidate---documind-api-{NUMBER}.{REGION}.run.app"   # make release-candidate GIT_SHA=<sha>: a NEW image on a no-traffic revision
FLIP = False   # True runs the traffic flip and the rollback below, on the live service; False prints the commands

print("kit:", KIT, "| API:", API_URL, "| gateway:", GATEWAY_URL, "| slm:", SLM_URL)


## Cell 1: The API, the gateway and the SLM, called the way the lane calls them


In [ ]:
import json, requests, time, subprocess, datetime
from google.cloud import storage

# THE API, THE GATEWAY AND THE SLM, CALLED THE WAY THE LANE CALLS THEM: one ID token per request, minted AS the roster
# member for the service's own URL (the kit mints it: documind_tools._id_token). Every service on the lane is behind
# Cloud Run IAM; there is no master key and no API key to paste.
def api(path: str, body: dict | None = None, base: str | None = None, timeout: int = 120, token: str | None = "member") -> tuple[int, dict | str]:
    """POST one API route (or a candidate revision's, with base=) - as documind-ui-sa by default, with token=None as
    nobody, or with a token minted as another account. Returns (status, json-or-text)."""
    url = (base or API_URL).rstrip("/")
    headers = {}
    if token == "member":
        headers["Authorization"] = f"Bearer {documind_tools._id_token(API_URL)}"     # the audience is the canonical URL
    elif token:
        headers["Authorization"] = f"Bearer {token}"
    r = requests.post(f"{url}{path}", json=body, headers=headers, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def api_get(path: str, base: str | None = None, timeout: int = 60) -> tuple[int, dict | str]:
    """GET one API route as the roster member: /version, /health."""
    url = (base or API_URL).rstrip("/")
    r = requests.get(f"{url}{path}", headers={"Authorization": f"Bearer {documind_tools._id_token(API_URL)}"}, timeout=timeout)
    try:
        return r.status_code, r.json()
    except ValueError:
        return r.status_code, r.text[:400]

def gateway(model: str, content: str, json_mode: bool = False, system: str | None = None, max_tokens: int = 200,
            timeout: int = 150) -> tuple[int, dict | str, dict]:
    """One OpenAI-compatible completion through the gateway (11.3), as the roster member. Returns (status, body, headers)."""
    msgs = ([{"role": "system", "content": system}] if system else []) + [{"role": "user", "content": content}]
    body = {"model": model, "messages": msgs, "max_tokens": max_tokens}
    if json_mode:
        body["response_format"] = {"type": "json_object"}
    r = requests.post(f"{GATEWAY_URL}/v1/chat/completions", json=body, timeout=timeout,
                      headers={"Authorization": f"Bearer {documind_tools._id_token(GATEWAY_URL)}"})
    try:
        return r.status_code, r.json(), dict(r.headers)
    except ValueError:
        return r.status_code, r.text[:400], dict(r.headers)

def service(name: str, region: str | None = None) -> dict:
    """A Cloud Run service as deployed: its env, labels, traffic, image and floor - read with gcloud, the way 10.3 did."""
    r = subprocess.run(["gcloud", "run", "services", "describe", name, "--region", region or REGION, "--project", PROJECT_ID,
                        "--format=json"], capture_output=True, text=True)
    if r.returncode != 0:
        return {}
    j = json.loads(r.stdout)
    c = j["spec"]["template"]["spec"]["containers"][0]
    return {"env": {e["name"]: e.get("value", "") for e in c.get("env", [])}, "image": c.get("image"),
            "labels": (j["metadata"].get("labels") or {}), "traffic": j.get("status", {}).get("traffic", []),
            "url": j.get("status", {}).get("url"), "sa": j["spec"]["template"]["spec"].get("serviceAccountName"),
            "annotations": (j["spec"]["template"]["metadata"].get("annotations") or {}),
            "min_instances": (j["spec"]["template"]["metadata"].get("annotations") or {}).get("autoscaling.knative.dev/minScale", "0")}

# The usage rows the API logs - the ONE shape every observability consumer reads (12.3, tenant_daily). On the lean
# lane they live in Cloud Logging; this reads the last few for a surface, newest first.
def usage_rows(minutes: int = 15, limit: int = 20, event: str = "query", service_name: str = "documind-api") -> list[dict]:
    since = (datetime.datetime.now(datetime.timezone.utc) - datetime.timedelta(minutes=minutes)).strftime("%Y-%m-%dT%H:%M:%SZ")
    r = subprocess.run(["gcloud", "logging", "read",
                        f'resource.type="cloud_run_revision" AND resource.labels.service_name="{service_name}" '
                        f'AND jsonPayload.event="{event}" AND timestamp>="{since}"',
                        "--project", PROJECT_ID, "--limit", str(limit), "--format=json"], capture_output=True, text=True)
    try:
        return [e["jsonPayload"] for e in json.loads(r.stdout or "[]")]
    except ValueError:
        return []

# THE TWENTY LINES THAT MATTER, FROM THE CLONE. Module 12's notebooks are where the kit's files come from (the heredoc
# cells at the end of each notebook are what extract_documind.py reads), so the walls stay there and the story reads
# the file the lane actually runs, around one line, with the file's length beside it.
def excerpt(rel: str, needle: str, before: int = 0, after: int = 14) -> str:
    lines = open(f"{KIT}/deploy/{rel}", encoding="utf-8").read().splitlines()
    i = next(n for n, l in enumerate(lines) if needle in l)
    lo, hi = max(0, i - before), min(len(lines), i + after)
    return f"# {rel}:{lo + 1}-{hi}  ({len(lines)} lines)\n" + "\n".join(lines[lo:hi])

def gcloud(*args: str) -> str:
    """One gcloud read, as the notebook's account, stdout only."""
    r = subprocess.run(["gcloud", *args, "--project", PROJECT_ID], capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else f"(gcloud: {r.stderr.strip()[:200]})"

gcs = storage.Client(project=PROJECT_ID)

sys.path.insert(0, f"{KIT}/deploy/evals")               # run_eval, usage_rows, judge
sys.path.insert(0, f"{KIT}/deploy/services/rag-api")    # the API's own modules, for the excerpts and the pure functions
print("helpers: api(), api_get(), gateway(), service(), usage_rows(), excerpt(), gcloud(); the kit's evals/ and rag-api/ on sys.path")


In [ ]:
from google.auth import impersonated_credentials
from google.auth.transport.requests import Request

def id_token_as(service_account: str, audience: str) -> str:
    """A Google ID token minted AS a service account, for one audience, with the email (4.8, 7.3) - the gate's outsider."""
    source, _ = google.auth.default()
    target = impersonated_credentials.Credentials(source_credentials=source, target_principal=service_account,
                                                  target_scopes=["https://www.googleapis.com/auth/cloud-platform"])
    idc = impersonated_credentials.IDTokenCredentials(target, target_audience=audience, include_email=True)
    idc.refresh(Request())
    return idc.token


## Cell 2: The offline gate, from the clone


In [ ]:
import run_eval

# THE OFFLINE GATE, FROM THE CLONE. run_eval.offline() is the first job of every PR and needs no credential: it checks
# the golden set against the corpus - every expected figure is falsifiable (present in the tenant's documents), every
# anchor resolves, every shape has its minimum rows (isolation 5, refusal 5, lookup 10, join 5). Red here blocks the
# merge before anything is built. The exit code is the mechanism; CI reads $?, not the report.
rc = run_eval.offline()
print("exit", rc)
assert rc == 0, "the offline gate is red: read the lines above"
print("thresholds:", run_eval.THRESHOLDS, "| minimum rows:", run_eval.MIN_ROWS)
golden = run_eval.load_golden()
print(len(golden), "rows;", {s: sum(1 for r in golden if r.get("shape") == s) for s in run_eval.MIN_ROWS})


### The pins, per image
The dry run's other checks read files, not the model: the requirements from the clone, a shared package pinned the same everywhere, and the two Dockerfiles it can parse but not build.


In [ ]:
# THE PINS, PER IMAGE - THE FILES THE DRY RUN READS. validate.py's checks 4 to 7 need no cloud: every dependency pinned,
# a package two images share pinned to the same version in both, a Dockerfile per service, every COPY source present in
# the context its Dockerfile assumes. The requirements files from the clone, listed here by hand and asserted against
# the disk (a new service must add itself to this list), the shared layer's, and the local lane's extra file - installed
# on top of chat's, never in an image. Then the two Dockerfiles on vendor bases (vLLM's, LiteLLM's), which the dry run
# parses and lints and cannot build.
import glob, re
REQS = ["services/rag-api/requirements.txt", "services/ingest/requirements.txt", "services/frontend/requirements.txt",
        "services/admin/requirements.txt", "services/mcp/requirements.txt", "services/chat/requirements.txt",
        "services/agent/requirements.txt", "services/gemma-vllm/requirements.txt", "services/litellm/requirements.txt",
        "services/slm/requirements.txt", "shared/requirements.txt"]
LOCAL = "services/chat/requirements-local.txt"
on_disk = sorted(os.path.relpath(p, f"{KIT}/deploy").replace(os.sep, "/") for p in glob.glob(f"{KIT}/deploy/services/*/requirements*.txt") + [f"{KIT}/deploy/shared/requirements.txt"])
assert on_disk == sorted(REQS + [LOCAL]), sorted(set(on_disk) ^ set(REQS + [LOCAL]))
pins = {}
for rel in REQS + [LOCAL]:
    lines = [l.strip() for l in open(f"{KIT}/deploy/{rel}", encoding="utf-8") if l.strip() and not l.startswith("#")]
    unpinned = [l for l in lines if "==" not in l]
    print(f"  {rel:42} {len(lines):>2} pins" + (f"   UNPINNED: {unpinned}" if unpinned else "") + ("   (on top of chat's, never in an image)" if rel == LOCAL else ""))
    for l in lines:
        m = re.match(r"([A-Za-z0-9_.-]+)(?:[[][^]]*[]])?==([^ ;#]+)", l)
        if m and rel != LOCAL:
            pins.setdefault(m.group(1).lower(), {})[rel] = m.group(2)
shared = {name: v for name, v in pins.items() if len(v) > 1}
drift = {name: v for name, v in shared.items() if len(set(v.values())) > 1}
print(f"{chr(10)}{len(shared)} packages shared by two or more images; pinned differently: {drift or 'none'}")
assert not drift, drift
for rel in ("services/gemma-vllm/Dockerfile", "services/litellm/Dockerfile"):
    text = open(f"{KIT}/deploy/{rel}", encoding="utf-8").read()
    print(f"{chr(10)}# {rel}")
    print(chr(10).join(l for l in text.splitlines() if l.startswith(("FROM", "COPY", "RUN", "ENTRYPOINT", "CMD"))))
print()
print("the same eleven files, and these two Dockerfiles, are what check 5 (pins) and check 7 (copy-paths) read on every push")


### The check that judges the tests instead of the model
This is `run_eval.py`'s first check, small enough to run here. Then run the real one on the real corpus: `python deploy/evals/run_eval.py`.


In [ ]:
# The check that judges the golden set instead of the model.
#
# This is deploy/evals/run_eval.py's first check, small enough to run here. Three
# tenants, three notice periods, deliberately different - because a corpus where
# every handbook says 60 days CANNOT demonstrate isolation. The leaked answer
# would also be the correct one.
CORPUS = {
    'acme':   'NP-03 Notice period: 60 days. EXP-12 Travel cap: Rs 40,000.',
    'zeta':   'NP-03 Notice period: 30 days. EXP-12 Travel cap: Rs 25,000.',
    'globex': 'MSA-04 Termination: 120 days notice. No HR policy.',
}


def audit(row: dict) -> list:
    """Can this row still go red for the right reason?"""
    own = CORPUS[row['tenant']].lower()
    problems = []
    for want in row.get('must_contain', []):
        if want.lower() not in own:
            problems.append(f"must_contain {want!r} is not in {row['tenant']}'s "
                            f"corpus - the row can ONLY fail")
    for never in row.get('must_not_contain', []):
        if never.lower() in own:
            problems.append(f"must_not_contain {never!r} IS {row['tenant']}'s own "
                            f"correct answer - the row fails when nothing is wrong")
        elif not any(never.lower() in t.lower()
                     for k, t in CORPUS.items() if k != row['tenant']):
            problems.append(f"must_not_contain {never!r} is in no other tenant's "
                            f"corpus - there is nothing to leak, so it is decoration")
    return problems


ROWS = [
    ('the row as committed',
     {'tenant': 'zeta', 'must_contain': ['25,000'], 'must_not_contain': ['40,000']}),
    ('somebody loosened must_contain to make CI green',
     {'tenant': 'zeta', 'must_contain': ['25,001'], 'must_not_contain': ['40,000']}),
    ("must_not_contain points at the tenant's OWN answer",
     {'tenant': 'zeta', 'must_contain': ['25,000'], 'must_not_contain': ['25,000']}),
    ('must_not_contain names a figure nobody has',
     {'tenant': 'zeta', 'must_contain': ['25,000'], 'must_not_contain': ['99,999']}),
]
for label, row in ROWS:
    found = audit(row)
    print(f"  {'RED  ' if found else 'green'}  {label}")
    for p in found:
        print(f'           {p}')
print()
print('  All three failures are edits somebody makes with good intentions on a bad')
print('  day, and every one of them turns a red build green while looking like a')
print('  fix. That is what this check is for: it does not test the model at all,')
print('  it tests whether the test could still fail.')
print()
print('  Run the real one on the real corpus:  python deploy/evals/run_eval.py')


## Cell 3: The candidate, and the live gate on it


In [ ]:
# THE CANDIDATE, AND THE LIVE GATE ON IT. make release-candidate puts the image make build pushed (api:<sha>) on a
# revision tagged candidate that takes no traffic; its /version names the sha. run_eval.live() is the gate the workflow
# runs against it - 64 rows, five thresholds, the outsider's 403s counted with the kit's tokens in the environment -
# and its exit code decides whether a person is even asked. Ten minutes on a cold candidate; the cell runs it only
# when a candidate exists, and prints what it would run otherwise.
st_l, live_v = api_get("/version")
st_c, cand_v = api_get("/version", base=CANDIDATE_URL)
print("live     :", live_v)
print("candidate:", cand_v if st_c == 200 else f"HTTP {st_c} - none tagged. Cloud Shell: make build SERVICES_lean=api GIT_SHA=<sha> && make release-candidate GIT_SHA=<sha>")
if st_c == 200:
    print("differs in:", [k for k in live_v if live_v.get(k) != cand_v.get(k)] or "nothing (an env-flip candidate, or the same image)")
    os.environ["DOCUMIND_ID_TOKEN"] = documind_tools._id_token(API_URL)
    os.environ["DOCUMIND_OUTSIDER_TOKEN"] = id_token_as(OUTSIDER_SA, API_URL)
    os.environ["DOCUMIND_USER_EMAIL"] = MEMBER_SA
    t0 = time.time()
    rc = run_eval.live(CANDIDATE_URL)
    print(f"\nexit {rc} after {time.time() - t0:.0f} s - {'the candidate clears: a person may promote it' if rc == 0 else 'red: the candidate stays where it is, taking no traffic'}")
else:
    print("would run: run_eval.live(CANDIDATE_URL) with DOCUMIND_ID_TOKEN / DOCUMIND_OUTSIDER_TOKEN minted as the roster member and the outsider (make eval-live API=<candidate>)")
# A REINDEX IS A RELEASE (12 September 2026, deploy/INDEXING.md): the same gate, scoped to the rows that cite one document -
# run_eval.live(url, source="hr_policy_2026.md"), make eval-live SOURCE=<name> API=<candidate> - judges a re-issued document on
# a candidate in minutes; a `version` row that cites a retired figure blocks on its own, and a threshold with no rows in
# scope is reported, not judged. The offline half runs before the upload (make reindex), so the rows move with the document.
print("scoped:", [r["id"] for r in run_eval.load_golden() if run_eval.in_scope(r, "hr_policy_2026.md", run_eval.load_corpus())],
      "- the rows a reindex of the handbook is judged on")


### Red, green, and the only thing CI reads


In [ ]:
# What the gate looks like when it works.
#
# A threshold, a run, and an exit code. The exit code is the whole mechanism -
# CI does not read your report, it reads $?.
# These are the real numbers, copied from deploy/evals/run_eval.py.
THRESHOLDS = {'answerable_rate': 0.80,      # of rows the corpus CAN answer, how many did
              'citation_rate': 0.95,        # of answered rows, how many carried a citation
              'must_contain_rate': 0.85,    # ...and held the expected figure
              'refusal_rate': 0.90,         # of rows it CANNOT answer, how many were refused
              'isolation_403_rate': 1.00}   # not 0.99. One leak is the whole product.


def gate(results: dict) -> tuple[int, list[str]]:
    """Returns (exit_code, reasons). Non-zero blocks the merge."""
    failures = [f'{k}: {results[k]:.0%} < {v:.0%}'
                for k, v in THRESHOLDS.items() if results.get(k, 0) < v]
    return (1 if failures else 0), failures


RUNS = [
    ('main today',
     {'answerable_rate': .87, 'citation_rate': .97, 'must_contain_rate': .93,
      'refusal_rate': .95, 'isolation_403_rate': 1.0}),
    ('PR: prompt drops citations',
     {'answerable_rate': .86, 'citation_rate': .61, 'must_contain_rate': .90,
      'refusal_rate': .95, 'isolation_403_rate': 1.0}),
    ('PR: membership loosened',
     {'answerable_rate': .88, 'citation_rate': .96, 'must_contain_rate': .92,
      'refusal_rate': .95, 'isolation_403_rate': 0.4}),
    ('PR: model stops saying "I cannot"',
     {'answerable_rate': .91, 'citation_rate': .97, 'must_contain_rate': .94,
      'refusal_rate': .20, 'isolation_403_rate': 1.0}),
]
for label, r in RUNS:
    code, why = gate(r)
    print(f'  exit {code}  {label:34} {"; ".join(why) if why else "all thresholds met"}')
print()
print('  Three red runs, three different regressions. Rows 3 and 4 are the ones')
print('  worth staring at: in both, every answer-QUALITY score is fine or better,')
print('  and the pipeline still stops.')
print()
print('  Row 4 is why refusal_rate exists at all. The first version of this runner')
print('  only sent the ANSWERABLE rows, so a model that confidently invented an')
print('  answer to every unanswerable question scored a clean 100%. Five refusal')
print('  rows and four of the five isolation rows were never asked. A gate only')
print('  measures what it actually sends.')
print()
print('  The exit code IS the mechanism. CI does not read your report; it reads')
print('  the number the process returned. A runner that prints FAILED in red and')
print('  exits 0 is a runner that has never blocked anything.')


### The gate that could not go red


In [ ]:
# The gate, and whether it can actually go red.
#
# The plan says: a PR with a bad prompt must be BLOCKED. Fine. But a gate that
# cannot fail is a green light with extra steps, so check that first.
#
# DocuMind's golden set has isolation rows: ask tenant A a question, assert the
# answer must_not_contain tenant B's figure. Here is why that row is nearly
# incapable of failing.
def retrieve_is_filtered() -> str:
    return ('retriever.py opens with a HARD predicate - Vector Search restricts '
            'on tenant_id, and the Firestore fallback is .where("tenant_id","==",t). '
            'No chunk from another tenant ever reaches the generator.')


print('  Why iso-* rows pass by construction:')
print('   ', retrieve_is_filtered())
print()
print('  So must_not_contain can only fail if the model INVENTS another tenant\'s')
print('  exact figure from nothing in its context. That is a hallucination')
print('  lottery, and it returns green almost always. It is not a test of')
print('  isolation; it is a test of luck.')
print()
# The leg that CAN leak is the other one.
def can_this_leak(leg: str) -> tuple[bool, str]:
    if leg == 'tenant -> chunks':
        return False, 'a hard predicate in retriever.py; nothing to test at the prompt layer'
    if leg == 'identity -> tenant':
        return True, ('enforce_membership(user_email, tenant_id) against the Firestore '
                      'roster - a LOOKUP, and lookups can be wrong')
    return False, 'unknown leg'


for leg in ('tenant -> chunks', 'identity -> tenant'):
    leaks, why = can_this_leak(leg)
    print(f'  {leg:20} can leak: {str(leaks):5}  {why}')
print()
print('  So the eval gate that earns its place asserts on the SECOND leg: give')
print('  the runner a user who is not on tenant B\'s roster and require a 403,')
print('  not an answer. That test fails the day somebody loosens')
print('  enforce_membership, which is a thing that actually happens.')
print()
print('  Keep the must_not_contain rows - they are cheap and they would catch a')
print('  genuinely catastrophic regression. Just do not call them the gate.')


## Cell 4: The key you did not create, read back


In [ ]:
# THE KEY YOU DID NOT CREATE, READ BACK. GitHub's runner proves who it is with an OIDC token; the provider in wif.tf
# accepts it only when three claims hold - the repository, the branch (var.deploy_ref), the actor - and lets it act as
# sa-documind-cicd, which holds exactly the roles a release needs and, since 10 September, may mint the two eval
# identities' tokens (cicd_eval_tokens). Nothing in the repository's secrets; the condition is the credential.
pool, provider = "documind-github", "github-oidc"
print(gcloud("iam", "workload-identity-pools", "providers", "describe", provider, f"--workload-identity-pool={pool}", "--location=global",
             "--format=value(attributeCondition)") or "(the provider is not created yet: wif.tf applies with make plan / the apply)")
print()
cicd = f"sa-documind-cicd@{PROJECT_ID}.iam.gserviceaccount.com"
raw = gcloud("projects", "get-iam-policy", PROJECT_ID, "--format=json")
try:
    roles = sorted(b["role"] for b in json.loads(raw).get("bindings", []) if f"serviceAccount:{cicd}" in b.get("members", []))
except ValueError:
    roles = [raw[:120]]
print(f"{cicd.split('@')[0]} holds:", roles)
for target in ("documind-ui-sa", "documind-outsider-sa"):
    pol = gcloud("iam", "service-accounts", "get-iam-policy", f"{target}@{PROJECT_ID}.iam.gserviceaccount.com", "--format=json")
    try:
        can = any(f"serviceAccount:{cicd}" in b.get("members", []) and "TokenCreator" in b["role"] for b in json.loads(pol).get("bindings", []))
    except ValueError:
        can = pol[:80]
    print(f"  may mint {target}'s tokens (the gate's two identities): {can}")
print()
print(excerpt("terraform/wif.tf", "attribute_condition", 0, 8))


### The thing you are not going to create


In [ ]:
# The thing you are not going to create.
#
# The old way: make a service-account JSON, paste it into a GitHub secret, and
# hope. That file is a credential with NO EXPIRY that every workflow in the repo
# can read, that leaves no trace when copied, and that nobody rotates.
#
# The new way: GitHub already signs a short-lived OIDC token describing WHO is
# running WHAT. Workload Identity Federation trades that token for a Google
# credential. Nothing is stored anywhere.
COMPARE = [
    ('what is stored in GitHub',  'a JSON private key',      'nothing'),
    ('lifetime',                  'until somebody rotates it', '~1 hour'),
    ('who can read it',           'every workflow in the repo', 'n/a'),
    ('if it leaks',               'valid until revoked',     'already expired'),
    ('proves which repo ran it',  'no',                      'yes, in the token'),
]
print(f'  {"":28} {"SA key":>26} {"WIF":>18}')
for q, old, new in COMPARE:
    print(f'  {q:28} {old:>26} {new:>18}')
print()
print('  The last row is the one that matters and the one people miss. A key')
print('  says "somebody who has this key". A token says "the main branch of')
print('  this repository, in this workflow, at this moment". You can write')
print('  rules against the second and you cannot against the first.')


### Not needing a key is not the same as not being able to make one


In [ ]:
# Not needing a key and not being ABLE to make one are different things.
#
# wif.tf removed the need. It did not remove the ability - and the ability is what
# survives a hurried Friday afternoon.
SCENARIOS = [
    ('the CI pipeline',            'uses WIF',              'no key involved'),
    ('a local debugging script',   'gcloud auth login',     'no key involved'),
    ('a colleague in a hurry',     'creates a key, emails it', 'KEY EXISTS FOREVER'),
    ('a vendor integration',       'asked for a JSON',      'KEY EXISTS FOREVER'),
]
print(f'  {"who":28} {"what they do":26} outcome')
for who, what, out in SCENARIOS:
    print(f'  {who:28} {what:26} {out}')
print()
print('  Rows 3 and 4 are the reason for an organization policy. Enforcing')
print('  iam.disableServiceAccountKeyCreation makes')
print('    gcloud iam service-accounts keys create ...')
print('  fail - and the error names the constraint, so the next person learns')
print('  WHERE the decision was made instead of filing a permissions bug.')
print()
print('  Google enforces this constraint BY DEFAULT on organizations created on')
print('  or after 3 May 2024. If yours is older, or somebody turned it off,')
print('  org_policy.tf turns it back on for this project.')
print()
print('  It is OFF by default in this kit, and that is about who is running it:')
print('  organization policy needs the project to belong to an ORGANIZATION. A')
print('  throwaway project under a personal account has none, and the resource')
print('  would fail the apply for a reason unrelated to anything you did.')


### Three gates on the token


In [ ]:
# Three gates. Two of them are the ones tutorials stop before.
#
# The provider decides which GitHub tokens it will even look at. Every claim you
# do not pin is a claim an attacker gets to choose.
CONDITION = [
    ('assertion.repository_id', "'1234567890'",
     'the IMMUTABLE numeric id. Names are re-registrable.'),
    ('assertion.repository', "'netsetos/agentic-ai-weekend-gcp'",
     'readable in logs and in the principalSet.'),
    ('assertion.ref', "'refs/heads/main'",
     'the BRANCH. Without it, any branch deploys.'),
]
for claim, value, why in CONDITION:
    print(f'  {claim:26} == {value:28} {why}')
print()

# Does a given token get in? This is the condition, evaluated.
def admitted(token: dict) -> tuple[bool, str]:
    if token.get('repository_id') != '1234567890':
        return False, 'wrong repository_id'
    if token.get('repository') != 'netsetos/agentic-ai-weekend-gcp':
        return False, 'wrong repository name'
    if token.get('ref') != 'refs/heads/main':
        return False, f"branch {token.get('ref')} is not main"
    return True, 'admitted'


TOKENS = [
    ('the real pipeline, on main',
     {'repository_id': '1234567890', 'repository': 'netsetos/agentic-ai-weekend-gcp',
      'ref': 'refs/heads/main'}),
    ('a colleague on a feature branch',
     {'repository_id': '1234567890', 'repository': 'netsetos/agentic-ai-weekend-gcp',
      'ref': 'refs/heads/add-deploy-step'}),
    ('someone who re-registered the freed NAME',
     {'repository_id': '9999999999', 'repository': 'netsetos/agentic-ai-weekend-gcp',
      'ref': 'refs/heads/main'}),
    ('an unrelated repository',
     {'repository_id': '5555555555', 'repository': 'someone/else',
      'ref': 'refs/heads/main'}),
]
for label, tok in TOKENS:
    ok, why = admitted(tok)
    print(f'  {"ADMITTED" if ok else "refused ":9} {label:42} {why}')
print()
print('  Row 2 is the one worth staring at. Without the ref pin, ANYONE who can')
print('  push a branch to your repository can add a workflow file and deploy to')
print('  production. That is most of your team, and every bot with write access.')
print()
print('  Row 3 is the reason for repository_id. Pin the NAME and a repository')
print('  you no longer own can still satisfy the gate.')


### What the deploy identity could actually reach


In [ ]:
# The deploy identity that could read every secret you have.
#
# This one shipped in the repo until today, and the comment above it said the
# opposite of what the code did.
#
# roles/iam.serviceAccountUser at PROJECT scope means: act as ANY service account
# in this project. Not the ones you deploy - all of them, including the ones
# somebody adds next year.
RUNTIME_SAS = {
    'documind-api-sa':    ['secretmanager.secretAccessor', 'datastore.user'],
    'documind-ui-sa':     ['secretmanager.secretAccessor'],
    'documind-ingest-sa': ['datastore.user', 'dlp.user'],
    'documind-admin-sa':  ['bigquery.dataViewer', 'datastore.user'],
    'sa-future-thing':    ['owner'],          # the one nobody has created yet
}


def reachable(actas_scope: str) -> set:
    """What the CI identity can do by submitting a build as somebody else."""
    if actas_scope == 'project':
        targets = RUNTIME_SAS                       # every SA, present and future
    else:
        targets = {k: v for k, v in RUNTIME_SAS.items() if k != 'sa-future-thing'}
    return {p for perms in targets.values() for p in perms}


for scope, label in (('project', 'actAs at PROJECT scope (what shipped)'),
                     ('enumerated', 'actAs per service account (fixed)')):
    perms = reachable(scope)
    print(f'  {label:40} reaches {len(perms)} permissions')
    for p in sorted(perms):
        print(f'      {p}')
    print()
print('  Both lists contain secretmanager.secretAccessor, and that is the honest')
print('  part: a pipeline that deploys a service running as X MUST be able to act')
print('  as X. You cannot fix that by tightening; it is what deploying means.')
print()
print('  What the fix buys is the BLAST RADIUS. The enumerated version cannot')
print('  reach sa-future-thing, because somebody has to add it to a list that a')
print('  reviewer reads. The project-scoped version picks it up silently, on the')
print('  day it is created, for ever.')


### Creating a release and running one are different jobs


In [ ]:
# Creating a release and running one are different jobs.
#
# Cloud Deploy names a service account as the EXECUTION identity for its render,
# deploy and verify steps. That is not the same as the identity that CREATES the
# release, even when it is the same account.
# Google's Cloud Deploy service-account page names THREE things for a Cloud Run
# target, and no more than three. Resist adding a fourth by reasoning.
NEEDED = {
    'clouddeploy.releaser':  'create a release',
    'clouddeploy.jobRunner': 'RUN the jobs - and it carries the bucket access too',
    'run.developer':         'actually actuate a Cloud Run service',
}
SHIPPED = {'cloudbuild.builds.editor', 'artifactregistry.writer',
           'clouddeploy.releaser', 'iam.serviceAccountUser'}

print(f'  {"role":26} {"present?":>9}  what it buys')
for role, why in NEEDED.items():
    print(f'  {role:26} {"yes" if role in SHIPPED else "MISSING":>9}  {why}')
print()
missing = [r for r in NEEDED if r not in SHIPPED]
print(f'  {len(missing)} of {len(NEEDED)} were missing. The release would have been')
print('  created successfully and then died at rollout - as a permission error')
print('  naming a service account, which is the single most confusing failure')
print('  in this whole pipeline because the account named is the one you chose.')
print()
print('  Two roles are NOT on that list, and the first draft of this lesson had')
print('  both: storage.objectAdmin and logging.logWriter. They sound obviously')
print('  needed - Cloud Deploy does stage manifests in a bucket, the jobs do log -')
print('  and jobRunner already covers the first while the service agent covers the')
print('  second.')
print()
print('  storage.objectAdmin at PROJECT scope would reach every bucket here,')
print('  including the uploads bucket of customer documents and the audit bucket')
print('  under a five-year retention lock. Adding a role because a service')
print('  "touches storage" is the same reflex that produced the project-scoped')
print('  actAs above. Read the documented list; do not derive one.')


## Cell 5: The workflow, parsed


In [ ]:
import yaml

# THE WORKFLOW, PARSED. documind-cd.yml is workflow_dispatch with a profile input. On lean the release-lean job is
# four make targets - build, release-candidate, eval-live on the candidate - and promote-lean waits for the production
# environment's reviewers before make promote. No gcloud deploy on lean: Cloud Deploy, staging and the canary are the
# full profile's job (release), gated by the same input. Read from the clone so the steps here are the steps that run.
wf = yaml.safe_load(open(f"{KIT}/.github/workflows/documind-cd.yml", encoding="utf-8"))
trigger = wf.get(True) or wf.get("on")
print("inputs:", {k: (v.get("options"), v.get("default")) for k, v in trigger["workflow_dispatch"]["inputs"].items()})
for job in ("eval-gate", "release-lean", "promote-lean", "release"):
    j = wf["jobs"][job]
    print(f"\n{job}:  if={j.get('if', '-')}  needs={j.get('needs', '-')}  environment={j.get('environment', '-')}")
    for s in j["steps"]:
        run = (s.get("run") or s.get("uses") or "").strip().splitlines()
        print(f"   - {s.get('name', s.get('uses'))[:44]:44} {run[-1][:80] if run else ''}")
lean_runs = " ".join(s.get("run", "") for s in wf["jobs"]["release-lean"]["steps"] + wf["jobs"]["promote-lean"]["steps"])
assert "gcloud deploy" not in lean_runs and "make release-candidate" in lean_runs and "make promote" in lean_runs
print("\nthe person: environment 'production' with required reviewers - the workflow stops there until one approves")


## Cell 6: Rollback is a traffic flip, not a build


In [ ]:
# ROLLBACK IS A TRAFFIC FLIP, NOT A BUILD. make promote moves traffic to the newest revision; make rollback moves it
# back to the one before, which never went away. Both are one gcloud call and about fifteen seconds; GET /version says
# which sha is answering afterwards. FLIP=False prints the commands with the revisions they would move between;
# FLIP=True runs them on the live service - the flip and then the rollback, timed, so the lane ends where it began.
revs = gcloud("run", "revisions", "list", "--service", "documind-api", "--region", REGION, "--sort-by=~metadata.creationTimestamp",
              "--format=value(metadata.name,status.conditions[0].status)", "--limit", "3").splitlines()
serving = [t for t in service("documind-api")["traffic"] if t.get("percent")]
print("newest revisions:", [r.split()[0] for r in revs])
print("serving now     :", [(t.get("revisionName"), t.get("percent")) for t in serving])
prev = revs[1].split()[0] if len(revs) > 1 else "PREV"
cmds = [("promote", ["run", "services", "update-traffic", "documind-api", "--region", REGION, "--to-latest"]),
        ("rollback", ["run", "services", "update-traffic", "documind-api", "--region", REGION, "--to-revisions", f"{prev}=100"])]
if not FLIP:
    for label, args in cmds:
        print(f"  would run ({label}): gcloud {' '.join(args)} --project {PROJECT_ID}")
    print("  then GET /version on the live URL after each - set FLIP = True to run them (the lane ends where it began)")
else:
    for label, args in cmds:
        t0 = time.time()
        out = gcloud(*args, "--quiet")
        st, ver = api_get("/version")
        print(f"  {label:9} {time.time() - t0:5.1f} s  -> /version {ver.get('git_sha')}  {out[:80]}")
print()
print(excerpt("Makefile", "rollback: guard-project", 0, 4, ))


## Cell 7: The full profile's pipeline, from the clone
The table below is that pipeline - Cloud Deploy, staging, a canary - as the first version drew it. On the lane it is one variable away; the lean release above is the same three gates without the second target.


In [ ]:
# THE FULL PROFILE'S PIPELINE, FROM THE CLONE. clouddeploy.tf declares two targets in one project - staging, then prod
# with require_approval - and a delivery pipeline; the workflow's release job creates a release, waits for the staging
# rollout, runs the live gate against staging, and promotes to prod where a canary takes 10% first. All of it is
# count = local.full ? 1 : 0, none of it is on the lane, and the first version's caveat still stands: two targets in
# one project separate nothing but the name and the approval. The real version is a second project - one variable.
print(excerpt("terraform/clouddeploy.tf", 'resource "google_clouddeploy_target" "prod"', 0, 12)); print()
print(excerpt("terraform/clouddeploy.tf", "google_clouddeploy_delivery_pipeline", 0, 8))
print("\non this lane:", gcloud("deploy", "delivery-pipelines", "list", "--region", REGION, "--format=value(name)") or "no delivery pipeline (lean)")


In [ ]:
# The whole thing, in order.
STAGES = [
    ('PR opened',        'run_eval.py            (offline)',       'no credentials; red blocks the merge'),
    ('merge to main',    'gcloud builds submit',                   'image tagged with the SHA'),
    ('release created',  'gcloud deploy releases create',          'one artefact, two targets'),
    ('staging',          'deploys unattended',                     'the workflow waits for SUCCEEDED'),
    ('verify',           'run_eval.py --api-url  (live)',          'the FIRST job needing a credential'),
    ('prod',             'WAITS for a human',                      'require_approval = true'),
    ('canary',           '10% of traffic, then the rest',          'and GET /version says which'),
    ('if it is wrong',   'traffic flip to the previous revision',  'under 2 minutes'),
]
print(f'  {"stage":18} {"what happens":42} why it is there')
for a, b, c in STAGES:
    print(f'  {a:18} {b:42} {c}')
print()
print('  THREE gates, and they are different KINDS of gate. Two are mechanical:')
print('  they run, they score, they exit non-zero. The third is a person.')
print('  Neither kind substitutes for the other - a machine cannot judge whether')
print('  now is a good time to deploy, and a person cannot re-read 59 golden')
print('  answers on every pull request.')
print()
print('  Now be precise about the keyless argument, because the loose version is')
print('  wrong. The live eval is NOT the first step needing a credential - build')
print('  and release authenticate before it. What is true, and is the point, is')
print('  that nothing before the MERGE needs one: rows 1 and 2 run on a GitHub')
print('  runner with no access to anything.')
print()
print('  So the demand for a credential appears exactly when you decide to check')
print('  quality against a real deployment. A service-account JSON in a repository')
print('  secret is the easy way to supply it, and step 1 is about not taking it.')


### One honest caveat


In [ ]:
# One honest caveat about this pipeline.
#
# staging and prod are two Cloud Deploy targets - and in this repo they point at
# the SAME project and the SAME region.
TARGETS = {
    'documind-staging': ('${project_id}', '${region}', 'require_approval: not set -> false'),
    'documind-prod':    ('${project_id}', '${region}', 'require_approval: true'),
}
for name, (proj, reg, approval) in TARGETS.items():
    print(f'  {name:20} project={proj:14} region={reg:10} {approval}')
print()
print('  So the only thing separating staging from production here is the NAME')
print('  and the approval step. That is enough to teach the shape of a pipeline,')
print('  and it is not enough for real money: a bad migration in staging is a bad')
print('  migration in prod, because it is the same database.')
print()
print('  Say this out loud rather than letting a learner infer isolation that is')
print('  not there. The real version is a second project, and the change is one')
print('  variable - which is the exercise at the bottom of this lesson.')


### And what the pipeline itself costs


In [ ]:
# What the pipeline itself costs, which is less than people expect.
#
# Verified on cloud.google.com/deploy/pricing, 5 Sept 2026.
PIPELINES = [
    ('documind (staging + prod)', 2, 'FIRST multi-target pipeline'),
    ('a second product,   2 targets', 2, 'second multi-target pipeline'),
    ('a one-target experiment',    1, 'single target'),
]
USD_INR = 85


def monthly_fee(targets: int, is_first_multi: bool) -> float:
    """Single-target pipelines carry no management fee. The first ACTIVE
    multi-target pipeline per billing account is free; each further one is $5."""
    if targets < 2:
        return 0.0
    return 0.0 if is_first_multi else 5.0


first_used = False
total = 0.0
for name, targets, note in PIPELINES:
    is_first = targets >= 2 and not first_used
    if is_first:
        first_used = True
    fee = monthly_fee(targets, is_first)
    total += fee
    print(f'  {name:32} {targets} target(s)  ${fee:5.2f}/mo   {note}')
print(f'  {"":32}             {"-" * 6}')
print(f'  {"management fee, total":32}             ${total:5.2f}/mo  '
      f'= Rs {total * USD_INR:,.0f}')
print()
print('  So DocuMind\'s delivery pipeline is FREE: it is the first active')
print('  multi-target pipeline on the account. "Active" means at least one')
print('  release or rollout was created that month - a pipeline nobody used')
print('  costs nothing at all.')
print()
print('  What you DO pay for is underneath: Cloud Build minutes for the image,')
print('  Cloud Storage for the rendered manifests, and Cloud Audit Logs. None')
print('  of them are large here, and none of them are zero. The honest line for')
print('  a review is "the delivery pipeline is free, the builds are not".')


## Cell 8: Where these files live


In [ ]:
# Where this file goes - and the bug that would have eaten it.
#
# deploy/ is GENERATED from the Module 12 notebooks. A notebook writes a heredoc;
# extract_documind.py decides where it lands. Until today that decision had three
# branches: .tf, .sql, and services/<name>/.
#
# A GitHub Actions workflow is none of those. So the heredoc for documind-cd.yml
# was FOUND by the collector, dropped by the placer, and counted as neither
# placed nor unresolved - and `extract --check` reported GREEN over a notebook
# that had written nothing at all.
DESTINATIONS = {
    'wif.tf':           'deploy/terraform/wif.tf',
    'tenant_daily.sql': 'deploy/terraform/sql/tenant_daily.sql',
    'documind-cd.yml':  '.github/workflows/documind-cd.yml   <- NEW',
    'run_eval.py':      'deploy/evals/run_eval.py            <- NEW',
    'run-service.yaml': 'deploy/run-service.yaml             <- NEW',
    'skaffold.yaml':    'UNPLACED - reported, not dropped    <- NEW',
}
for f, d in DESTINATIONS.items():
    print(f'  {f:20} -> {d}')
print()
print('  Two changes, and the second matters more than the first. Adding the')
print('  workflow destination fixes one file. Making an unplaceable file a')
print('  REPORTED line rather than a silent skip fixes every file anybody adds')
print('  from now on - including the ones nobody has thought of.')
print()
print('  This is the rule the whole module runs on: to change a deployed file,')
print('  edit the NOTEBOOK and re-extract. Editing deploy/ directly works until')
print('  the next extraction quietly reverts you.')


In [ ]:
# Not executed here - this is the runbook.
COMMANDS = [
    ('make eval',                      'the offline gate, ~1 second, no credentials'),
    ('make eval-live API=https://...', 'the live gate against a deployment'),
    ('make dryrun',                    'extract --check + validate + eval'),
    ('python deploy/extract_documind.py --check',
     'the tree still matches these notebooks'),
]
for cmd, what in COMMANDS:
    print(f'  {cmd:36} {what}')
print()
print('  And the one to run after ANY edit to a heredoc above:')
print('    python deploy/extract_documind.py && python deploy/extract_documind.py --check')
print()
print('  If --check is green but a file you expected did not appear, look for an')
print('  UNPLACED line in the extract output. Before this lesson there was no such')
print('  line - the file was simply dropped.')


## Where this goes
- **12.8** runs the seven smokes as one and turns the lane off - the last two things a release day does.
- **4.8** is the gate itself; **12.6** is the candidate this lesson judged.

## ✅ Lesson 12.7 complete
- ✅ The offline gate run from the clone: falsifiable, anchored, covered; exit 0
- ✅ The pins per image read from the clone and asserted consistent; the two vendor-based Dockerfiles printed
- ✅ The candidate's /version beside the live one; the live gate run on it with the kit's two identities
- ✅ The provider's condition and the deploy account's roles read back; the two token-creator grants
- ✅ The workflow's lean jobs parsed: build, candidate, gate, a person, promote - and no gcloud deploy
- ✅ The flip and the rollback as commands with their revisions; the full profile's pipeline read from the clone


## The files this lesson owns
Below are the eight heredocs the extractor places: the version file, the Workload Identity provider (its branch pin a variable since 10 September, and the two token-creator grants for the gate's identities), the Cloud Deploy pipeline (full profile), the workflow (a profile input; `release-lean` and `promote-lean` beside the full `release` job), the organisation policy that makes keyless enforceable, the Cloud Run manifest, the Cloud Build config and the gate itself. The story above ran the gate from the clone and read the rest back from the lane.


In [ ]:
# Rollback is a traffic flip, not a build.
#
# The slowest possible rollback is "revert the commit and wait for CI". The
# fastest is to point traffic at the revision that was already working, which is
# still sitting there.
# Note the command in row 2. There is NO bare `gcloud deploy rollback` - the
# group is `gcloud deploy targets`, and the target is a positional argument.
# Getting that wrong costs you the two minutes the gate allows.
STEPS = [
    ('git revert, rebuild, redeploy',   12 * 60, 'a fresh build of code you already had'),
    ('gcloud deploy targets rollback',  90,      'promotes the previous release'),
    ('gcloud run update-traffic',       15,      'that revision never went away'),
]
print(f'  {"approach":34} {"seconds":>8}  note')
for how, secs, note in STEPS:
    print(f'  {how:34} {secs:>8}  {note}')
print()
print('  In full, the middle one is:')
print('      gcloud deploy targets rollback documind-prod'
      ' --delivery-pipeline=documind --region=$REGION')
print()
print('  The gate for this lesson is a rollback in under two minutes, and only')
print('  the bottom two clear it. Practise the traffic flip before you need it -')
print('  at 3am nobody reads a runbook they have never run.')
print()
# And you have to be able to SEE which one is live. This endpoint already exists:
# 12.2 wrote it, whole. Today is the day it earns its keep.
VERSION = '''@app.get("/version")          # services/rag-api/main.py, shipped since 12.2
def version():
    return {"model_backend": settings.model_backend,
            "generator_model": settings.generator_model,
            "prompt": f"{settings.prompt_id}@{settings.prompt_version}",
            "retrieval_mode": settings.retrieval_mode,
            "git_sha": os.environ.get("GIT_SHA", "unknown")}
'''
print(VERSION)
print('Without it, "did the rollback actually work?" is answered by watching a')
print('dashboard and hoping. With it, the answer is one curl and a string compare -')
print('and the same triple appears on every log line, so the graph and the endpoint')
print('cannot disagree.')


In [ ]:
WIF_TF = r'''
# Keyless CI. UNOWNED: lesson 12.7 teaches this; it is stood up early because the
# 16 Sept masterclass demo (D4.1) needs a pipeline that exists.
#
# The point is the ABSENCE of a key. A service-account JSON in a GitHub secret is a
# credential with no expiry that every workflow in the repo can read; WIF exchanges a
# short-lived GitHub OIDC token for one, scoped to one repository.
resource "google_iam_workload_identity_pool" "github" {
  workload_identity_pool_id = "documind-github"
  display_name              = "GitHub Actions"
}

resource "google_iam_workload_identity_pool_provider" "github" {
  workload_identity_pool_id          = google_iam_workload_identity_pool.github.workload_identity_pool_id
  workload_identity_pool_provider_id = "github-oidc"
  display_name                       = "GitHub OIDC"

  attribute_mapping = {
    "google.subject"          = "assertion.sub"
    "attribute.repository"    = "assertion.repository"
    # The IMMUTABLE one. Repository and org NAMES are re-registrable after a
    # rename, transfer or deletion; the numeric id is not.
    "attribute.repository_id" = "assertion.repository_id"
    "attribute.ref"           = "assertion.ref"
  }

  # Three gates, and the first two are the ones tutorials stop before.
  #
  #   repository_id  pins the repo that cannot be impersonated by re-registering
  #                  a name somebody released. Pinning the NAME alone means
  #                  whoever claims "netsetos/agentic-ai-weekend-gcp" after a rename gets
  #                  a GitHub-signed token that satisfies the condition.
  #   ref            pins the BRANCH. Without it, any workflow on any branch or
  #                  tag in the repo gets full deploy credentials - so anyone who
  #                  can push a branch can add a workflow file and deploy.
  #   repository     kept for readability in logs and in the principalSet.
  attribute_condition = join(" && ", [
    "assertion.repository_id == '${var.github_repository_id}'",
    "assertion.repository == '${var.github_repository}'",
    "assertion.ref == '${var.deploy_ref}'",
  ])

  oidc { issuer_uri = "https://token.actions.githubusercontent.com" }
}

# Only this repository may impersonate the deploy identity - by immutable id, for
# the same reason the condition uses it. A principalSet keyed on the NAME is a
# second place the reclamation trick would work.
resource "google_service_account_iam_member" "cicd_wif" {
  service_account_id = google_service_account.cicd.name
  role               = "roles/iam.workloadIdentityUser"
  member = format(
    "principalSet://iam.googleapis.com/%s/attribute.repository_id/%s",
    google_iam_workload_identity_pool.github.name,
    var.github_repository_id,
  )
}

output "wif_provider" {
  description = "workload_identity_provider for the GitHub action"
  value       = google_iam_workload_identity_pool_provider.github.name
}
output "cicd_service_account" { value = google_service_account.cicd.email }

# Which ref may deploy. main by default; the demo weeks may point it at the lane's branch (Module 12, decision D5) and
# back again - one variable, in the plan output, never a second provider.
variable "deploy_ref" {
  type    = string
  default = "refs/heads/main"
}

# The lean release job (documind-cd.yml, release-lean) runs the live gate as the roster member and as the outsider -
# make eval-live impersonates both - so the deploy identity may mint their tokens, and nothing else's: two accounts,
# enumerated, for the same reason cicd_actas is.
resource "google_service_account_iam_member" "cicd_eval_tokens" {
  for_each           = { ui = google_service_account.ui.name, outsider = google_service_account.outsider.name }
  service_account_id = each.value
  role               = "roles/iam.serviceAccountTokenCreator"
  member             = "serviceAccount:${google_service_account.cicd.email}"
}
'''

with open('wif.tf', 'w') as f: f.write(WIF_TF)
print('wif.tf:', len(WIF_TF.splitlines()), 'lines')


In [ ]:
CLOUDDEPLOY_TF = r'''
# Staging, then a canary in prod. UNOWNED: lesson 12.7 teaches it.
#
# A canary is not a slower deploy - it is a deploy you can stop. 10% of traffic on the
# new revision, a pause, then the rest; `gcloud deploy targets rollback documind-prod`
# at any point. (The `targets` group is not optional - there is no bare
# `gcloud deploy rollback`, and finding that out during an incident is the wrong time.)
resource "google_clouddeploy_target" "staging" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  location = var.region
  name     = "documind-staging"
  run { location = "projects/${var.project_id}/locations/${var.region}" }
  execution_configs {
    usages            = ["RENDER", "DEPLOY", "VERIFY"]
    service_account   = google_service_account.cicd.email
    execution_timeout = "3600s"
  }
}

resource "google_clouddeploy_target" "prod" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  location = var.region
  name     = "documind-prod"
  run { location = "projects/${var.project_id}/locations/${var.region}" }
  # A human says yes before production. The eval gate in 12.7 runs before this point;
  # this is the second gate, and it is a person.
  require_approval = true
  execution_configs {
    usages            = ["RENDER", "DEPLOY", "VERIFY"]
    service_account   = google_service_account.cicd.email
    execution_timeout = "3600s"
  }
}

resource "google_clouddeploy_delivery_pipeline" "documind" {
  count = local.full ? 1 : 0   # the full profile only (variables.tf)
  location = var.region
  name     = "documind"
  serial_pipeline {
    stages {
      target_id = google_clouddeploy_target.staging[0].name
      profiles  = ["staging"]
    }
    stages {
      target_id = google_clouddeploy_target.prod[0].name
      profiles  = ["prod"]
      strategy {
        canary {
          runtime_config {
            cloud_run { automatic_traffic_control = true }
          }
          canary_deployment {
            percentages = [10]
            # FALSE, deliberately. verify runs a skaffold `verify` stanza, and the release is
            # cut with --from-run-manifest, whose generated skaffold config has none. Leaving
            # this true makes the canary fail at its verify phase - AFTER 10% of production
            # traffic is already on the new revision, which is the worst moment to find out.
            #
            # The check still happens: documind-cd.yml waits for the staging rollout and then
            # runs deploy/evals/run_eval.py against it, before prod is promoted at all. Flip
            # this to true the day a skaffold verify stanza exists to run.
            verify = false
          }
        }
      }
    }
  }
}

output "delivery_pipeline" { value = one(google_clouddeploy_delivery_pipeline.documind[*].name) }
'''

with open('clouddeploy.tf', 'w') as f: f.write(CLOUDDEPLOY_TF)
print('clouddeploy.tf:', len(CLOUDDEPLOY_TF.splitlines()), 'lines')


In [ ]:
CD_YML = r'''
# DocuMind delivery. MANUAL ONLY (workflow_dispatch), deliberately: this repository is a
# content repo with no GCP project behind it, and a deploy that fires on push is not
# something to switch on as a side effect. In the application repo the trigger is
#
#   on:
#     pull_request:          # the eval gate runs, and red blocks the merge
#     push:
#       branches: [main]     # merged and green -> build, release, staging, then a human
#
# and nothing else about this file changes. The gate is the jobs below, not the trigger.
#
# Keyless: no service-account JSON anywhere. GitHub mints a short-lived OIDC token
# describing which repository, which workflow and which branch is running; Workload
# Identity Federation exchanges it for a Google credential that expires in about an hour.
# See deploy/terraform/wif.tf, which pins three claims:
#
#   repository_id  the IMMUTABLE numeric id. Names are re-registrable after a rename or
#                  transfer, so a condition pinned to the NAME can be satisfied by
#                  whoever claims that name next.
#   repository     kept for readability in logs and in the principalSet.
#   ref            refs/heads/main. Without it, anyone able to push a BRANCH could add a
#                  workflow file and deploy to production.
#
# A workflow_dispatch run from any other branch therefore fails at the auth step. That is
# the branch gate working, not a misconfiguration.
name: documind-cd

on:
  workflow_dispatch:
    inputs:
      image_tag:
        description: Image tag to release (defaults to the commit SHA)
        required: false
        type: string
      profile:
        description: lean = the lane's release (a candidate revision, the live gate on it, a person, a traffic flip); full = Cloud Deploy
        required: false
        type: choice
        options: [lean, full]
        default: lean

permissions:
  contents: read
  id-token: write          # REQUIRED for OIDC; without it auth fails with no token

jobs:
  # ---------------------------------------------------------------- gate 1: mechanical
  # The offline half of the eval gate. It judges the GOLDEN SET, not the model: that every
  # assertion could still fail, that every must_retrieve anchor resolves, and that the
  # isolation rows have not been deleted to make a red build green. No credentials, so it
  # also runs on every pull request from documind-dryrun.yml.
  eval-gate:
    runs-on: ubuntu-latest
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - name: Eval gate (offline)
        run: python deploy/evals/run_eval.py

  # ---------------------------------------------------------------- the lane's release (lean)
  # Module 10's seam as a pipeline, keyless. The same image tagged with the SHA; a candidate revision of documind-api
  # that takes no traffic (make release-candidate); the live gate on it - 64 rows, five thresholds, as the roster
  # member and as the outsider, whose tokens the deploy identity may mint (wif.tf); then a PERSON, through the
  # production environment's required reviewers; then the traffic flip, which is also the rollback (make rollback).
  # No Cloud Deploy verb on this path: the previous revision never went away, and GET /version says which is live.
  release-lean:
    if: ${{ inputs.profile == 'lean' }}
    needs: eval-gate
    runs-on: ubuntu-latest
    env:
      PROJECT: ${{ vars.GCP_PROJECT }}
      REGION: ${{ vars.GCP_REGION }}
    steps:
      - uses: actions/checkout@v4
      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"
      - id: auth
        uses: google-github-actions/auth@v3
        with:
          workload_identity_provider: ${{ vars.WIF_PROVIDER }}
          service_account: ${{ vars.CICD_SERVICE_ACCOUNT }}
      - uses: google-github-actions/setup-gcloud@v3
      - name: Resolve tag
        env:
          IMAGE_TAG: ${{ inputs.image_tag }}
        run: echo "TAG=${IMAGE_TAG:-$GITHUB_SHA}" >> "$GITHUB_ENV"
      - name: Build the API image
        working-directory: deploy
        run: make build PROJECT="$PROJECT" REGION="$REGION" SERVICES_lean=api GIT_SHA="$TAG"
      - name: A candidate revision, no traffic
        working-directory: deploy
        run: make release-candidate PROJECT="$PROJECT" REGION="$REGION" GIT_SHA="$TAG"
      - name: The live gate, on the candidate
        working-directory: deploy
        run: |
          NUMBER=$(gcloud projects describe "$PROJECT" --format='value(projectNumber)')
          make eval-live PROJECT="$PROJECT" REGION="$REGION" API="https://candidate---documind-api-$NUMBER.$REGION.run.app"

  promote-lean:
    needs: release-lean
    runs-on: ubuntu-latest
    environment: production      # the person: the environment's required reviewers approve, then traffic moves
    env:
      PROJECT: ${{ vars.GCP_PROJECT }}
      REGION: ${{ vars.GCP_REGION }}
    steps:
      - uses: actions/checkout@v4
      - id: auth
        uses: google-github-actions/auth@v3
        with:
          workload_identity_provider: ${{ vars.WIF_PROVIDER }}
          service_account: ${{ vars.CICD_SERVICE_ACCOUNT }}
      - uses: google-github-actions/setup-gcloud@v3
      - name: Traffic to the candidate (the release; the previous revision never went away)
        working-directory: deploy
        run: make promote PROJECT="$PROJECT" REGION="$REGION"

  release:
    if: ${{ inputs.profile == 'full' }}
    needs: eval-gate          # red above means this job never starts
    runs-on: ubuntu-latest
    env:
      PROJECT: ${{ vars.GCP_PROJECT }}
      REGION: ${{ vars.GCP_REGION }}
    steps:
      - uses: actions/checkout@v4

      - uses: actions/setup-python@v5
        with:
          python-version: "3.12"

      # v3 (Node 24). The inputs are unchanged from v2 for the WIF+SA path.
      - id: auth
        uses: google-github-actions/auth@v3
        with:
          workload_identity_provider: ${{ vars.WIF_PROVIDER }}
          service_account: ${{ vars.CICD_SERVICE_ACCOUNT }}

      - uses: google-github-actions/setup-gcloud@v3

      # Through the ENVIRONMENT, not into the script text. `${{ }}` is substituted before
      # bash parses the line, so an input containing a backtick or $( ) would execute.
      - name: Resolve tag
        env:
          IMAGE_TAG: ${{ inputs.image_tag }}
        run: echo "TAG=${IMAGE_TAG:-$GITHUB_SHA}" >> "$GITHUB_ENV"

      # `gcloud builds submit` has NO --file flag, and --tag requires a Dockerfile at the
      # root of the source it uploads. rag-api imports shared/ (the answer contract, the IAP
      # verifier), so its context is deploy/ - and deploy/ has no root Dockerfile, on purpose.
      # cloudbuild.yaml names the Dockerfile explicitly; every service builds this way.
      - name: Build
        working-directory: deploy
        run: |
          gcloud builds submit --config=cloudbuild.yaml \
            --substitutions=_IMAGE="$REGION-docker.pkg.dev/$PROJECT/documind/rag-api:$TAG" .

      - name: Create a release
        working-directory: deploy
        run: |
          gcloud deploy releases create "rel-$TAG" \
            --delivery-pipeline=documind \
            --region="$REGION" \
            --from-run-manifest=run-service.yaml \
            --images="rag-api=$REGION-docker.pkg.dev/$PROJECT/documind/rag-api:$TAG"

      # Cloud Deploy's own verify step would be the textbook home for this. It runs a
      # skaffold `verify` stanza, and --from-run-manifest generates a skaffold config that
      # has no such stanza - so the wait and the check live here instead, in the open, where
      # a learner can read them. clouddeploy.tf therefore sets verify = false: declaring a
      # verify phase with nothing to run fails the canary at its verify step.
      - name: Wait for the staging rollout
        run: |
          for i in $(seq 1 60); do
            # `|| true` on purpose: the step runs under set -e, and one flaky API call
            # should not abort a twenty-minute wait.
            state=$(gcloud deploy rollouts list \
              --delivery-pipeline=documind --release="rel-$TAG" \
              --region="$REGION" --filter="targetId=documind-staging" \
              --format="value(state)" --limit=1 2>/dev/null || true)
            echo "rollout state: ${state:-PENDING}"
            case "$state" in
              SUCCEEDED) exit 0 ;;
              FAILED|CANCELLED|HALTED) exit 1 ;;
            esac
            sleep 20
          done
          echo "staging rollout did not finish in 20 minutes" >&2
          exit 1

      # ------------------------------------------------------------ gate 2: mechanical
      # The LIVE half, against staging.
      #
      # Be precise about the keyless argument, because the loose version is wrong: this is
      # NOT the first step needing a credential - builds submit and releases create both
      # authenticated, above, in this same job. What is true, and is the actual point, is
      # that nothing before the MERGE needs one. documind-dryrun.yml runs the offline gate
      # on every pull request with no Google credential at all, and the first thing that
      # does need one is this post-merge job. A service-account JSON in a repository secret
      # would have been the easy way to give it one.
      #
      # The threshold that matters here is isolation_403_rate: it asks as somebody who is
      # NOT on the tenant roster and requires a 403. See run_eval.check_isolation() for
      # why must_not_contain is reported but is not the gate.
      # The token is minted HERE, not by the auth step. google-github-actions/auth only
      # populates `steps.auth.outputs.id_token` when `token_format: id_token` is set, and that
      # needs an audience - which for Cloud Run is the service URL, unknown until the describe
      # below. Referencing the output without token_format yields an EMPTY string, and
      # run_eval.py's `os.environ.get(...) or None` turns that into "send no Authorization
      # header": the gate would call an unauthenticated endpoint and pass for the wrong reason.
      - name: Eval gate (live, against staging)
        run: |
          URL=$(gcloud run services describe documind-api \
            --region="$REGION" --format="value(status.url)")
          DOCUMIND_ID_TOKEN=$(gcloud auth print-identity-token --audiences="$URL")
          export DOCUMIND_ID_TOKEN
          python deploy/evals/run_eval.py --api-url "$URL"

      # Only now does prod come into view - and it stops for a person. require_approval
      # is set on the prod target in clouddeploy.tf, so this promotion queues rather than
      # deploying, then canaries at 10%.
      - name: Promote to prod (queues for approval)
        run: |
          gcloud deploy releases promote \
            --release="rel-$TAG" \
            --delivery-pipeline=documind \
            --region="$REGION" \
            --to-target=documind-prod

      # Rolling back is a traffic flip, not a build. `gcloud deploy targets rollback`
      # promotes the previous release - note the `targets` group; there is no bare
      # `gcloud deploy rollback`. The fastest path of all is
      #   gcloud run services update-traffic documind-api --to-revisions=PREV=100
      # because that revision never went away. Check which one is live with GET /version.
      - name: Where it went
        run: |
          echo "Release rel-$TAG: staging deployed and verified."
          echo "Prod is queued for approval, then canaries at 10%."
          echo "Roll back with:"
          echo "  gcloud deploy targets rollback documind-prod \\"
          echo "    --delivery-pipeline=documind --region=$REGION"
'''

with open('documind-cd.yml', 'w') as f: f.write(CD_YML)
print('documind-cd.yml:', len(CD_YML.splitlines()), 'lines')


In [ ]:
ORG_POLICY_TF = r'''
# The policy that makes "keyless" true instead of aspirational. Lesson 12.7.
#
# wif.tf removes the NEED for a service-account key. This removes the ABILITY to make
# one. Those are different things, and only the second survives a hurried afternoon:
# the pipeline being keyless does not stop somebody creating a key for a local script,
# emailing it to a colleague, and leaving it in a Slack thread for three years.
#
# iam.disableServiceAccountKeyCreation is a boolean constraint. Enforced, `gcloud iam
# service-accounts keys create` fails with a message naming this constraint - which is
# the useful part, because it tells the next person where the decision was made instead
# of looking like a permissions bug.
#
# Google enforces it BY DEFAULT on organizations created on or after 3 May 2024. If
# yours is older, or you turned it off, this turns it back on for this project.

variable "enforce_no_sa_keys" {
  type        = bool
  default     = false
  description = <<-EOT
    Enforce iam.disableServiceAccountKeyCreation on this project.

    Defaults to FALSE, and that default is about who is running this rather than about
    whether it is a good idea. Organization policy needs the project to belong to an
    ORGANIZATION and the caller to hold roles/orgpolicy.policyAdmin. A learner's
    throwaway project under a personal account has no organization at all, and this
    resource would fail the apply for a reason unrelated to anything they did.

    On a real project: set it to true. That is the whole exercise.
  EOT
}

resource "google_org_policy_policy" "no_sa_keys" {
  count  = var.enforce_no_sa_keys ? 1 : 0
  name   = "projects/${var.project_id}/policies/iam.disableServiceAccountKeyCreation"
  parent = "projects/${var.project_id}"

  spec {
    # inherit_from_parent is not set: this project's rule stands on its own, so a
    # loosened folder-level policy cannot quietly re-enable key creation here.
    rules {
      enforce = "TRUE"
    }
  }
}

output "sa_keys_disabled" {
  value       = var.enforce_no_sa_keys
  description = "TRUE means no service-account key can be created in this project."
}
'''

with open('org_policy.tf', 'w') as f: f.write(ORG_POLICY_TF)
print('org_policy.tf:', len(ORG_POLICY_TF.splitlines()), 'lines')


In [ ]:
RUN_SERVICE_YAML = r'''
# The Cloud Run service manifest Cloud Deploy renders. Lesson 12.7.
#
# WHY THIS FILE EXISTS. `gcloud deploy releases create` needs a skaffold configuration: that is
# how Cloud Deploy knows what to render for each target. This kit has no skaffold.yaml and does
# not want one - a Cloud Run pipeline does not need the rest of skaffold - so the release is cut
# with `--from-run-manifest=run-service.yaml`, which makes gcloud generate the skaffold config
# for you from this single file.
#
# Without it the pipeline fails at the FIRST credentialed step, before any rollout exists, with
# a message about a missing skaffold configuration. That is not a subtle failure, but it is one
# you only meet on the day you first run the pipeline for real.
#
# NOTE THE MISSING TRAFFIC STANZA. There is deliberately no `traffic:` block here. Cloud Deploy
# owns traffic during a canary - it sets 10%, waits, then the rest - and a traffic stanza in the
# manifest fights it: every rollout would slam 100% to the new revision and the canary would be
# decorative. The absence is the feature.
apiVersion: serving.knative.dev/v1
kind: Service
metadata:
  name: documind-api
  annotations:
    run.googleapis.com/ingress: all
spec:
  template:
    metadata:
      annotations:
        autoscaling.knative.dev/minScale: "0"     # 11.4's lesson: never leave this at 1
        autoscaling.knative.dev/maxScale: "10"
    spec:
      serviceAccountName: documind-api-sa
      containerConcurrency: 8
      timeoutSeconds: 300
      containers:
        # The image is a PLACEHOLDER name, not a tag. `gcloud deploy releases create --images`
        # binds it: --images="rag-api=REGION-docker.pkg.dev/PROJECT/documind/rag-api:SHA".
        # The literal string here must match the key on the left of that = sign.
        - image: rag-api
          ports:
            - containerPort: 8080
          env:
            - name: AUTH_MODE
              value: iap
            # The two legs of shared/iap.identity() (12.8, gap G4). SELF_URL is this service's
            # own URL - the audience an agent, run_eval.py or smoke.py mints an ID token for
            # (7.3); Cloud Run's URLs are deterministic, so it is known before the first deploy.
            # IAP_AUDIENCE is every SURFACE whose forwarded assertion this service accepts,
            # and no other: the UI and the chat service, by project NUMBER, not id.
            - name: SELF_URL
              value: https://documind-api-PROJECT_NUMBER.us-central1.run.app
            - name: IAP_AUDIENCE
              value: /projects/PROJECT_NUMBER/locations/us-central1/services/documind-ui,/projects/PROJECT_NUMBER/locations/us-central1/services/documind-chat
          resources:
            limits:
              cpu: "2"
              memory: 2Gi
'''

with open('run-service.yaml', 'w') as f: f.write(RUN_SERVICE_YAML)
print('run-service.yaml:', len(RUN_SERVICE_YAML.splitlines()), 'lines')


In [ ]:
CLOUDBUILD_YAML = r'''
# Build the rag-api image from the deploy/ context. Lessons 12.2 and 12.7.
#
# WHY A CONFIG AND NOT `gcloud builds submit --tag`. --tag requires a Dockerfile at the ROOT of
# the uploaded source. rag-api now imports deploy/shared/ (the answer contract, the IAP
# verifier, the tenant roster), so its build context has to be deploy/ - and deploy/ has no
# Dockerfile at its root, on purpose: eight services live under it. This file names the
# Dockerfile explicitly and builds from `.` = deploy/.
#
#   gcloud builds submit --config=cloudbuild.yaml \
#     --substitutions=_IMAGE=REGION-docker.pkg.dev/PROJECT/documind/rag-api:TAG .
#
# chat, ingest and admin build from the same context; any of them is this config with a
# different Dockerfile (12.8's chat deploy passes _DOCKERFILE=services/chat/Dockerfile).
steps:
  - name: gcr.io/cloud-builders/docker
    args:
      - build
      - -f
      - ${_DOCKERFILE}
      - -t
      - ${_IMAGE}
      - .
images:
  - ${_IMAGE}
substitutions:
  _IMAGE: us-central1-docker.pkg.dev/documind-ai-YOUR-ID/documind/rag-api:dev
  _DOCKERFILE: services/rag-api/Dockerfile
options:
  logging: CLOUD_LOGGING_ONLY
'''

with open('cloudbuild.yaml', 'w') as f: f.write(CLOUDBUILD_YAML)
print('cloudbuild.yaml:', len(CLOUDBUILD_YAML.splitlines()), 'lines')


In [ ]:
RUN_EVAL_PY = r'''
#!/usr/bin/env python3
"""The eval gate. Exit code 0 merges; anything else blocks.

    python deploy/evals/run_eval.py                    # OFFLINE - no credentials, no cost
    python deploy/evals/run_eval.py --api-url URL      # LIVE    - needs a deployment
    python deploy/evals/run_eval.py --api-url URL --source hr_policy_2026.md   # LIVE, scoped to the rows citing one document

Two modes, because the gate has two halves that fail for different reasons and one of
them must run on every pull request.

OFFLINE is the half that runs in CI here (documind-dryrun.yml). It never calls Google.
It checks the golden set itself:

    falsifiable   every must_contain value really is in that tenant's corpus, and every
                  must_not_contain value really is in ANOTHER tenant's and not in its
                  own. build_golden.py asserts this when it WRITES the file; this
                  asserts it over the file that is actually committed, which is the one
                  CI runs. Those differ the moment somebody edits golden.jsonl by hand
                  to make a red build go green - which is the single most common way an
                  eval suite rots.
                  A `version` row (12 September 2026) is the ledger's: its must_contain is
                  the CURRENT version's figure, its must_not_contain a figure only a retired
                  version under evals/demo holds. It is what forces the rows to move with
                  the document, in the same commit: re-issue the handbook without moving
                  lk-06 and vr-01 and this check turns red before anything is deployed.
    anchors       every must_retrieve anchor is findable. evals/README.md left one
                  decision to this lesson: 12.5 mints chunk ids as {tenant}:{sha256}#{i}, which
                  contains neither the document slug nor the clause id, so the runner
                  matches each anchor against filename + text joined together.
                  NOTE the scope: this runs OFFLINE, over the corpus. The live half does
                  not score must_retrieve at all - /v1/query returns citations, not the
                  raw retrieved set, so recall cannot be measured from outside the
                  service. Scoring it live needs a debug field on the response; until
                  that exists, saying so here is better than implying coverage.
    coverage      the isolation and refusal rows still exist. Deleting a failing test
                  is the other common way a suite rots, and it looks like a green build.

LIVE is the half that needs a running service, and it is why this pipeline had to be
keyless. Be precise about that claim, because the loose version is wrong: this is not
the first STEP needing a credential - the build and the release authenticate before it,
in the same job. What is true is that nothing before the MERGE needs one. The offline
half above runs on every pull request with no Google credential at all, and the first
thing that does need one is this. A service-account JSON in a repository secret would
have been the easy way to provide it, and 12.7 is the lesson about not doing that.

LIVE sends EVERY row, not only the answerable ones - see live(). Scoring only the
answerable subset silently skipped all five refusal rows and four of the five isolation
rows, which is how a gate reports green over tests it never ran.

--source scopes the live half to the rows that cite one document (a slug in must_retrieve,
or the row's own `source`): a reindex is a release (deploy/INDEXING.md), and this is its
gate on a candidate - minutes, not the ten of the full set. A threshold with no rows in
scope is reported and not judged; a `version` row that cites a retired figure is a stale
answer and blocks on its own.

The threshold that matters live is NOT must_not_contain. Read WHY in check_isolation().
"""
from __future__ import annotations

import argparse
import json
import os
import re
import sys
import urllib.error
import urllib.request

HERE = os.path.dirname(os.path.abspath(__file__))
CORPUS = os.path.join(HERE, "corpus")
DEMO = os.path.join(HERE, "demo")          # the rehearsal's retired versions: what a version row's must_not_contain lives in
GOLDEN = os.path.join(HERE, "golden.jsonl")

# What a green run has to clear. Numbers, in one place, so raising a threshold is a
# reviewable line and not a conversation.
THRESHOLDS = {
    "answerable_rate": 0.80,    # of rows the corpus CAN answer, how many did
    "citation_rate": 0.95,      # of answered rows, how many carried a citation
    "must_contain_rate": 0.85,  # of answered rows, how many held the expected figure
    "refusal_rate": 0.90,       # of rows the corpus CANNOT answer, how many were refused
    "isolation_403_rate": 1.00, # not 0.99. One leak is the whole product.
}
MIN_ROWS = {"isolation": 5, "refusal": 5, "lookup": 10, "join": 5, "version": 1}


TEXT_FILES = (".md", ".txt")


def load_corpus() -> dict[str, dict[str, str]]:
    """{tenant: {path_relative_to_corpus: text}}. Text is lower-cased once.

    Text files only. Since the real documents arrived (evals/real_sources.json) a tenant
    directory also holds PDFs - the objects upload.sh pushes - and, for each one with a
    text layer, the .md mirror fetch_real.py extracted beside it. The mirror is what a golden
    row is verified against; the PDF is bytes, and a scanned Act (posh_act_2013) has no
    mirror at all, so nothing here can assert on it."""
    out: dict[str, dict[str, str]] = {}
    for tenant in sorted(os.listdir(CORPUS)):
        d = os.path.join(CORPUS, tenant)
        if not os.path.isdir(d):
            continue
        out[tenant] = {}
        for fn in sorted(os.listdir(d)):
            p = os.path.join(d, fn)
            if os.path.isfile(p) and fn.endswith(TEXT_FILES):
                out[tenant][fn] = open(p, encoding="utf-8").read()
    return out


def load_demo() -> str:
    """Every retired version the rehearsal keeps under evals/demo, lower-cased: the text a version row's
    must_not_contain must live in, or the row asserts a staleness nothing could produce."""
    if not os.path.isdir(DEMO):
        return ""
    return "\n".join(open(os.path.join(DEMO, fn), encoding="utf-8").read()
                     for fn in sorted(os.listdir(DEMO)) if fn.endswith(TEXT_FILES)).lower()


def load_golden() -> list[dict]:
    with open(GOLDEN, encoding="utf-8") as f:
        return [json.loads(line) for line in f if line.strip()]


def haystack(corpus: dict, tenant: str) -> str:
    """THE MATCHING RULE, offline. Filenames joined with contents, because an anchor may
    be a document slug (hr_policy_2026 - only in the path) or a clause id (EXP-12 - only
    in the text).

    The same rule WOULD run over chunk_id + source_uri + text on a real retrieval, which
    is the decision evals/README.md asked 12.7 to make. It does not run live today:
    /v1/query returns citations, not the retrieved set, so recall is not observable from
    outside the service. Saying that here beats implying a coverage that does not exist."""
    docs = corpus.get(tenant, {})
    return "\n".join(f"{name}\n{text}" for name, text in docs.items()).lower()


def sources_of(row: dict, corpus: dict) -> set[str]:
    """The documents a row cites: the must_retrieve anchors that are document slugs in the row's tenant's corpus
    (a clause code is not a document), plus the row's own `source` when it names one."""
    slugs = {name.rsplit(".", 1)[0] for name in corpus.get(row["tenant"], {})}
    out = {a for a in row.get("must_retrieve", []) if a in slugs}
    if row.get("source"):
        out.add(row["source"].rsplit("/", 1)[-1].rsplit(".", 1)[0])
    return out


def in_scope(row: dict, source: str | None, corpus: dict) -> bool:
    if not source:
        return True
    return source.rsplit("/", 1)[-1].rsplit(".", 1)[0] in sources_of(row, corpus)


# ------------------------------------------------------------------ offline checks
def check_falsifiable(golden: list[dict], corpus: dict) -> list[str]:
    """A test that cannot fail is not a test. Prove each assertion could."""
    bad = []
    demo = load_demo()
    for row in golden:
        own = haystack(corpus, row["tenant"])
        # build_golden.py enforces these when it WRITES the file. Re-assert them over the
        # file that is actually committed: deleting an assertion is the cheapest way to
        # make a red row green, and it leaves the row in place looking like a test.
        if row["shape"] == "isolation" and not row.get("must_not_contain"):
            bad.append(f"{row['id']}: an isolation row with no must_not_contain asserts "
                       f"nothing about isolation")
        if row["answerable"] and not row.get("must_contain"):
            bad.append(f"{row['id']}: an answerable row with no must_contain accepts "
                       f"any answer at all")
        for want in row.get("must_contain", []):
            if want.lower() not in own:
                bad.append(f"{row['id']}: must_contain {want!r} is not in "
                           f"{row['tenant']}'s corpus - the row can only fail")
        if row["shape"] == "version":
            # The ledger's row. The figure it forbids must be ABSENT from the current version of its source
            # (or a correct answer fails) and PRESENT in a retired version under evals/demo (or nothing
            # stale could ever produce it). Re-issue the document without moving the row, and this is red.
            src = (row.get("source") or "").rsplit("/", 1)[-1]
            current = corpus.get(row["tenant"], {}).get(src, "").lower()
            if not src or not current:
                bad.append(f"{row['id']}: a version row must name a text document of its tenant as `source`")
            if not row.get("must_not_contain"):
                bad.append(f"{row['id']}: a version row with no must_not_contain asserts nothing about staleness")
            for never in row.get("must_not_contain", []):
                if never.lower() in current:
                    bad.append(f"{row['id']}: must_not_contain {never!r} IS in the current version of "
                               f"{src} - the row fails on a correct answer; move the row with the document")
                if never.lower() not in demo:
                    bad.append(f"{row['id']}: must_not_contain {never!r} is in no retired version under "
                               f"evals/demo - nothing stale holds it, so the row is decoration")
            continue
        for never in row.get("must_not_contain", []):
            if never.lower() in own:
                bad.append(f"{row['id']}: must_not_contain {never!r} IS in "
                           f"{row['tenant']}'s own corpus - the row fails on a "
                           f"correct answer")
            elsewhere = [t for t in corpus
                         if t != row["tenant"] and never.lower() in haystack(corpus, t)]
            if not elsewhere:
                bad.append(f"{row['id']}: must_not_contain {never!r} is in no OTHER "
                           f"tenant's corpus - there is nothing to leak, so the row "
                           f"is decoration")
    return bad


def check_anchors(golden: list[dict], corpus: dict) -> list[str]:
    """Every must_retrieve anchor must be findable under the matching rule above."""
    bad = []
    for row in golden:
        own = haystack(corpus, row["tenant"])
        for anchor in row.get("must_retrieve", []):
            if anchor.lower() not in own:
                bad.append(f"{row['id']}: anchor {anchor!r} matches nothing in "
                           f"{row['tenant']}'s corpus (slug? clause id? typo?)")
    return bad


def check_coverage(golden: list[dict]) -> list[str]:
    """Deleting the failing rows is not a fix, and it looks exactly like a green run."""
    counts: dict[str, int] = {}
    for row in golden:
        counts[row["shape"]] = counts.get(row["shape"], 0) + 1
    return [f"shape {shape!r}: {counts.get(shape, 0)} rows, expected at least {n}"
            for shape, n in MIN_ROWS.items() if counts.get(shape, 0) < n]


def offline() -> int:
    corpus, golden = load_corpus(), load_golden()
    print(f"  {len(golden)} golden rows over {len(corpus)} tenants, "
          f"{sum(len(v) for v in corpus.values())} documents\n")
    failures = []
    for name, found in (("falsifiable", check_falsifiable(golden, corpus)),
                        ("anchors", check_anchors(golden, corpus)),
                        ("coverage", check_coverage(golden))):
        print(f"  [{'FAIL' if found else 'PASS'}] {name}")
        for line in found:
            print(f"         {line}")
        failures += found
    print()
    if failures:
        print(f"  {len(failures)} problem(s). The golden set cannot judge the model "
              f"until it judges itself.")
        return 1
    print("  The golden set is sound. It can go red, and it still contains the rows "
          "that would.")
    return 0


# ------------------------------------------------------------- the figure, not the spelling
_SMALL = {"zero": 0, "one": 1, "two": 2, "three": 3, "four": 4, "five": 5, "six": 6, "seven": 7,
          "eight": 8, "nine": 9, "ten": 10, "eleven": 11, "twelve": 12, "thirteen": 13,
          "fourteen": 14, "fifteen": 15, "sixteen": 16, "seventeen": 17, "eighteen": 18,
          "nineteen": 19, "twenty": 20, "thirty": 30, "forty": 40, "fifty": 50, "sixty": 60,
          "seventy": 70, "eighty": 80, "ninety": 90}
_SCALE = {"hundred": 100, "thousand": 1000, "lakh": 100000, "crore": 10000000}
_WORD = "|".join(list(_SMALL) + list(_SCALE))
_NUMWORDS = re.compile(r"\b(?:(?:%s)(?:[\s-]+(?:%s))*)\b" % (_WORD, _WORD))


def _to_int(run: str) -> int:
    total = current = 0
    for w in re.split(r"[\s-]+", run):
        if w in _SMALL:
            current += _SMALL[w]
        elif w == "hundred":
            current = (current or 1) * 100
        elif w in _SCALE:
            total += (current or 1) * _SCALE[w]
            current = 0
    return total + current


def normalise(text: str) -> str:
    """Lower-case, thousands separators out, number words to digits - on BOTH sides of every
    live containment check, so "twelve weeks" and "12 weeks" are the same fact.

    The third live eval (7 Sept 2026) answered lk-23 with "300 or more" against a row that
    says "three hundred", and jn-08 with "12 weeks" against "twelve weeks": the right figure
    in the statute's own spelling, scored as wrong. must_contain measures the figure; this
    makes the check say so. Nothing else is relaxed - the words around the number must still
    match - and must_not_contain gets the same treatment, so a leak spelt "26 weeks" is
    caught where "twenty-six weeks" alone would have let it through.

    The second media eval (9 Sept 2026) added two more spellings of a right answer: "set-on and
    set-off" against a row that says "set on" (mm-02), and "5.2%" against "5.2 per cent" (mm-03,
    the figure the segment prompt had just learned to keep). A hyphen is a space and per cent is
    %, on both sides - the figure, not the typography."""
    t = text.lower().replace(",", "").replace("-", " ")
    for spelled in (" per cent", "per cent", " percent", "percent"):
        t = t.replace(spelled, "%")
    return _NUMWORDS.sub(lambda m: str(_to_int(m.group(0))), t)


# --------------------------------------------------------------------- live checks
def ask(api_url: str, question: str, tenant: str, email: str, token: str | None):
    """POST /v1/query. Returns (status, body). AUTH_MODE=dev accepts x-user-email;
    under IAP the identity comes from the assertion and the header is ignored."""
    body = json.dumps({"query": question, "tenant_id": tenant, "top_k": 6}).encode()
    req = urllib.request.Request(f"{api_url.rstrip('/')}/v1/query", data=body,
                                 method="POST")
    req.add_header("Content-Type", "application/json")
    req.add_header("x-user-email", email)
    if token:
        req.add_header("Authorization", f"Bearer {token}")
    try:
        with urllib.request.urlopen(req, timeout=90) as r:
            return r.status, json.loads(r.read())
    except urllib.error.HTTPError as e:
        return e.code, {}
    except Exception as e:
        return 0, {"error": type(e).__name__}


def check_isolation(api_url, golden, token, outsider_token=None) -> tuple[float, list[str]]:
    """The isolation gate, and it is NOT must_not_contain.

    retriever.py filters by tenant BEFORE anything reaches the generator - Vector
    Search restricts on tenant_id, and the Firestore fallback carries the same
    equality predicate. So no chunk from another tenant is ever in the context, and
    must_not_contain can only fail if the model invents another tenant's exact figure
    out of nothing. That is a hallucination lottery, and a gate that fires by chance
    is not a gate.

    The leg that can actually leak is the other one: identity -> tenant. That is
    enforce_membership() in auth.py, a Firestore lookup, and lookups can be loosened.
    So the gate asks as somebody who is NOT on the roster and requires a 403.
    """
    outsider = os.environ.get("DOCUMIND_OUTSIDER_EMAIL", "outsider@not-a-tenant.invalid")
    # Under IAP (AUTH_MODE=iap) the API ignores x-user-email: the caller IS the bearer token's
    # account (shared/iap.py's other leg). So the outsider has to be a second token, minted
    # for an account that may invoke the service and sits on no roster - `make eval-live`
    # uses documind-chat-sa. With one token every row would carry the same, rostered identity
    # and this gate could not turn red.
    rows = [r for r in golden if r["shape"] == "isolation"]
    got, bad = 0, []
    for row in rows:
        status, _ = ask(api_url, row["question"], row["tenant"], outsider, outsider_token or token)
        if status == 403:
            got += 1
        else:
            bad.append(f"{row['id']}: a non-member asking {row['tenant']} got "
                       f"{status}, expected 403")
    return (got / len(rows) if rows else 0.0), bad


def live(api_url: str, source: str | None = None) -> int:
    corpus = load_corpus()
    golden = [r for r in load_golden() if in_scope(r, source, corpus)]
    if source:
        print(f"  scoped to {source}: {len(golden)} row(s) cite it")
        if not golden:
            print("  no golden row cites this document - the gate has nothing to judge; add a row before the reindex")
            return 1
    token = os.environ.get("DOCUMIND_ID_TOKEN") or None
    outsider_token = os.environ.get("DOCUMIND_OUTSIDER_TOKEN") or None
    member = os.environ.get("DOCUMIND_USER_EMAIL", "eval@documind.in")
    if token and not outsider_token:
        print("  note: DOCUMIND_ID_TOKEN set without DOCUMIND_OUTSIDER_TOKEN - under IAP the "
              "isolation rows will ask as the same rostered account and cannot pass.")

    # EVERY row is sent, not just the answerable ones.
    #
    # The first version of this loop iterated `[r for r in golden if r["answerable"]]`, which
    # silently dropped all five refusal rows AND four of the five isolation rows (iso-02..05
    # are answerable:false). So the leak tests carrying ACME's invoice total, Form 16 date,
    # revenue and PAN were never asked, and a model that fabricated an answer to every
    # unanswerable question would have scored a clean 100%.
    answerable_rows = [r for r in golden if r["answerable"]]
    unanswerable_rows = [r for r in golden if not r["answerable"]]
    isolation_rows = [r for r in golden if r["shape"] == "isolation"]
    answered = cited = contained = refused = 0
    kind_rows = kind_hits = 0   # must_cite_kind (Module 9): reported beside the thresholds, not one of them
    leaks = []
    stale = []       # a version row that cited a retired figure: the ledger's promise broken, or a row that did not move
    misses = []      # every row that cost a point, with what the API said - the first live
                     # run printed five rates and left the eighteen refused rows to guesswork
    for row in golden:
        status, body = ask(api_url, row["question"], row["tenant"], member, token)
        if status != 200:
            misses.append(f"{row['id']:6} {row['shape']:8} {row['tenant']:7} HTTP {status}   | {row['question'][:70]}")
            continue
        text = normalise(body.get("answer") or "")

        # must_not_contain is checked on EVERY 200, whatever the row's shape. A leak in a
        # refusal row is the same leak; a retired figure in a version row is a stale answer.
        for never in row.get("must_not_contain", []):
            if normalise(never) in text:
                (stale if row["shape"] == "version" else leaks).append(f"{row['id']}: answer contained {never!r}")

        if row["answerable"]:
            if body.get("answerable"):
                answered += 1
                if body.get("citations"):
                    cited += 1
                want_kind = row.get("must_cite_kind")
                if want_kind:
                    # A figure or a segment citation, not only the right words: the media is
                    # in the corpus (make media, make ingest-corpus) or this row says it is not.
                    kind_rows += 1
                    kinds = sorted({c.get("kind", "text") for c in body.get("citations") or []})
                    if want_kind in kinds:
                        kind_hits += 1
                    else:
                        misses.append(f"{row['id']:6} {row['shape']:8} {row['tenant']:7} cited no {want_kind} (kinds {kinds}) | {row['question'][:60]}")
                if all(normalise(w) in text for w in row.get("must_contain", [])):
                    contained += 1
                else:
                    misses.append(f"{row['id']:6} {row['shape']:8} {row['tenant']:7} answered without {row.get('must_contain')} | {text[:80]!r}")
            else:
                misses.append(f"{row['id']:6} {row['shape']:8} {row['tenant']:7} REFUSED conf={body.get('confidence')} cites={len(body.get('citations') or [])} | {row['question'][:70]}")
        else:
            # The corpus cannot answer this. Saying so IS the right answer.
            if not body.get("answerable"):
                refused += 1
            else:
                misses.append(f"{row['id']:6} {row['shape']:8} {row['tenant']:7} ANSWERED (should refuse) | {text[:80]!r}")

    n = len(answerable_rows)
    u = len(unanswerable_rows)
    iso_rate, iso_bad = check_isolation(api_url, golden, token, outsider_token)
    scores = {
        "answerable_rate": answered / n if n else 0.0,
        "citation_rate": cited / max(answered, 1),
        "must_contain_rate": contained / max(answered, 1),
        "refusal_rate": refused / u if u else 0.0,
        "isolation_403_rate": iso_rate,
    }
    # A threshold with no rows behind it is not judged: a scoped run (--source) may hold no refusal or isolation
    # row, and 0 of 0 refused is not a failing rate - it is an absence, and it is printed as one.
    rows_behind = {"answerable_rate": n, "citation_rate": answered, "must_contain_rate": answered,
                   "refusal_rate": u, "isolation_403_rate": len(isolation_rows)}
    judged = {k for k in THRESHOLDS if rows_behind[k] > 0}
    missed = {k for k in judged if scores[k] < THRESHOLDS[k]}
    failures = [f"{k}: {scores[k]:.0%} < {THRESHOLDS[k]:.0%}" for k in sorted(missed)]

    print(f"  {len(golden)} rows ({len(answerable_rows)} answerable, {len(unanswerable_rows)} not) against {api_url}\n")
    for k, v in scores.items():
        tag = "[FAIL]" if k in missed else ("[PASS]" if k in judged else "[ -- ]")
        print(f"  {tag} {k:20} {v:6.1%}  (threshold {THRESHOLDS[k]:.0%}{'' if k in judged else '; no rows in scope'})")
    if kind_rows:
        print(f"  [info] {'media_kind_rate':20} {kind_hits / kind_rows:6.1%}  ({kind_hits}/{kind_rows} rows asked for a "
              f"figure or segment citation; not a threshold - 0 means the media is not ingested)")
    for line in iso_bad + leaks + stale:
        print(f"         {line}")
    if misses:
        print(f"\n  rows that cost a point ({len(misses)}):")
        for line in misses:
            print(f"    {line}")
    print()
    # A must_not_contain hit is not in the thresholds, on purpose - see
    # check_isolation. It is still printed, and it is still a stop-everything.
    if leaks:
        print("  A must_not_contain row fired. That should be near-impossible given "
              "the retrieval filter, which makes it MORE serious, not less.")
        return 2
    if stale:
        print("  A version row cited a RETIRED figure. Either the ledger served a version it should have retired, or "
              "the document was re-issued and the golden rows did not move with it. Blocked.")
        failures.append("stale: a retired version was cited")
    if failures:
        print(f"  Blocked: {'; '.join(failures)}")
        return 1
    print("  All thresholds met.")
    return 0


def main() -> int:
    ap = argparse.ArgumentParser(description=__doc__.split("\n")[0])
    ap.add_argument("--api-url", help="run the LIVE half against this deployment")
    ap.add_argument("--source", help="scope the live half to the rows citing this document (a name or slug); "
                                     "offline, list them")
    args = ap.parse_args()
    if args.api_url:
        print("== eval gate: LIVE ==")
        return live(args.api_url, args.source)
    print("== eval gate: OFFLINE (no credentials, no cost) ==")
    rc = offline()
    if args.source:
        corpus = load_corpus()
        rows = [r for r in load_golden() if in_scope(r, args.source, corpus)]
        print(f"\n  rows citing {args.source}: {len(rows)} - the scoped live gate judges these")
        for r in rows:
            print(f"    {r['id']:6} {r['shape']:9} {r['question'][:72]}")
    return rc


if __name__ == "__main__":
    sys.exit(main())
'''

with open('run_eval.py', 'w') as f: f.write(RUN_EVAL_PY)
print('run_eval.py:', len(RUN_EVAL_PY.splitlines()), 'lines')
